# Data Evolution Across the Medallion Architecture

This notebook traces how IoT telemetry data transforms as it flows through
**Bronze** (raw) → **Silver** (clean, enriched) → **Gold** (aggregated, ML-ready).

At each layer we examine:
- Record counts and how they change (dedup, corrupt-record removal)
- Schema evolution (new columns added by each layer)
- Data quality improvements (anomaly flags, z-scores, quality scores)
- Delta Lake lineage (version history, time travel)

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pyspark.sql.functions import (
    col, count, avg, stddev, countDistinct,
    min as spark_min, max as spark_max,
    sum as spark_sum, when, lit,
)
from delta.tables import DeltaTable

from pipeline.common.utils import load_config, get_spark_session

config = load_config()
spark = get_spark_session(config)

## 1. Pipeline Funnel: Record Counts Across Layers

The medallion architecture is a progressive refinement. Each layer intentionally
sheds bad data and adds analytical value.

In [ ]:
bronze_path = config["paths"]["delta_bronze"]
silver_path = config["paths"]["delta_silver"]
gold_base   = config["paths"]["delta_gold"]

bronze_df = spark.read.format("delta").load(bronze_path)
silver_df = spark.read.format("delta").load(silver_path)

bronze_count = bronze_df.count()
silver_count = silver_df.count()

gold_tables = {}
gold_total = 0
for t in ["device_summary", "anomaly_summary", "device_health", "ml_features"]:
    try:
        gdf = spark.read.format("delta").load(str(Path(gold_base) / t))
        c = gdf.count()
        gold_tables[t] = {"df": gdf, "count": c, "cols": len(gdf.columns)}
        gold_total += c
    except Exception:
        gold_tables[t] = {"df": None, "count": 0, "cols": 0}

dropped = bronze_count - silver_count
print(f"Bronze:  {bronze_count:,} rows  (raw ingestion)")
print(f"Silver:  {silver_count:,} rows  ({dropped:,} records removed by cleaning)")
print(f"Gold:    {gold_total:,} rows  (across {len(gold_tables)} aggregate tables)")
print(f"\nDrop rate: {dropped/bronze_count*100:.1f}% of raw records cleaned out")

In [ ]:
fig = go.Figure(go.Funnel(
    y=["Bronze (Raw)", "Silver (Clean)", "Gold (Aggregated)"],
    x=[bronze_count, silver_count, gold_total],
    textinfo="value+percent initial",
    marker=dict(color=["#cd7f32", "#c0c0c0", "#ffd700"]),
))
fig.update_layout(
    title="Pipeline Record Funnel",
    template="plotly_white", height=350,
)
fig.show()

## 2. Schema Evolution: Columns Added at Each Layer

Each layer enriches the schema with new derived columns while dropping
internal metadata that downstream consumers don't need.

In [ ]:
bronze_cols = set(bronze_df.columns)
silver_cols = set(silver_df.columns)

print("=== Bronze Schema ===")
print(f"  {len(bronze_cols)} columns: {sorted(bronze_cols)}")
print(f"\n=== Silver Schema ===")
print(f"  {len(silver_cols)} columns")
print(f"  Added by Silver:   {sorted(silver_cols - bronze_cols)}")
print(f"  Dropped from Bronze: {sorted(bronze_cols - silver_cols)}")

if gold_tables["ml_features"]["df"] is not None:
    gold_cols = set(gold_tables["ml_features"]["df"].columns)
    print(f"\n=== Gold ML Features Schema ===")
    print(f"  {len(gold_cols)} columns (wide feature vector)")
    print(f"  Sample columns: {sorted(list(gold_cols))[:10]}...")

In [ ]:
layers = ["Bronze", "Silver", "Gold (summary)", "Gold (health)", "Gold (ML)"]
col_counts = [
    len(bronze_cols),
    len(silver_cols),
    gold_tables["device_summary"]["cols"],
    gold_tables["device_health"]["cols"],
    gold_tables["ml_features"]["cols"],
]

fig = go.Figure(go.Bar(
    x=layers, y=col_counts,
    marker_color=["#cd7f32", "#c0c0c0", "#ffd700", "#daa520", "#b8860b"],
    text=col_counts, textposition="outside",
))
fig.update_layout(
    title="Column Count Evolution Across Layers",
    yaxis_title="Number of Columns",
    template="plotly_white", height=350,
)
fig.show()

## 3. Bronze Layer: Raw Ingestion

Bronze captures everything as-is, including corrupt records and duplicates.
This is intentional — nothing is lost at the raw layer.

In [ ]:
print("=== Bronze Data Profile ===")
bronze_df.select(
    count("*").alias("total_rows"),
    count(col("_corrupt_record")).alias("corrupt_records"),
    countDistinct("device_id").alias("distinct_devices"),
    countDistinct("_source_file").alias("source_files"),
    spark_min("_ingested_at").alias("earliest_ingestion"),
    spark_max("_ingested_at").alias("latest_ingestion"),
).show(truncate=False)

print("=== Numeric Field Ranges ===")
bronze_df.select(
    spark_min("temperature").alias("temp_min"),
    spark_max("temperature").alias("temp_max"),
    avg("temperature").alias("temp_avg"),
    spark_min("humidity").alias("hum_min"),
    spark_max("humidity").alias("hum_max"),
    spark_min("battery_level").alias("batt_min"),
    spark_max("battery_level").alias("batt_max"),
).show(truncate=False)

print("=== Sample Records ===")
bronze_df.select(
    "device_id", "timestamp", "temperature", "humidity",
    "pressure", "battery_level", "_corrupt_record",
).show(5, truncate=False)

## 4. Silver Layer: Governance and Enrichment

Silver applies the full quality pipeline:
1. Remove corrupt/unparseable records
2. Drop nulls on required fields (`device_id`, `timestamp`)
3. Deduplicate by `(device_id, timestamp)`
4. Clamp humidity to `[0, 100]`
5. Flag anomalies per sensor field
6. Compute z-scores against expected population
7. Score overall record quality

In [ ]:
print("=== Silver Data Profile ===")
silver_df.select(
    count("*").alias("total_rows"),
    countDistinct("device_id").alias("distinct_devices"),
    spark_sum(when(col("_is_anomaly"), lit(1)).otherwise(lit(0))).alias("anomaly_count"),
    avg("_quality_score").alias("avg_quality_score"),
    spark_min("_quality_score").alias("min_quality_score"),
).show(truncate=False)

print("=== Anomaly Breakdown ===")
silver_df.selectExpr(
    "sum(cast(_is_temp_anomaly as int)) as temperature_anomalies",
    "sum(cast(_is_humidity_anomaly as int)) as humidity_anomalies",
    "sum(cast(_is_pressure_anomaly as int)) as pressure_anomalies",
    "sum(cast(_is_anomaly as int)) as total_anomalies",
).show(truncate=False)

In [ ]:
quality_pdf = silver_df.select("_quality_score", "_is_anomaly").toPandas()

clean = quality_pdf[quality_pdf["_is_anomaly"] == False]["_quality_score"]
anom  = quality_pdf[quality_pdf["_is_anomaly"] == True]["_quality_score"]

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=clean, name="Clean Records", marker_color="#2ecc71",
    opacity=0.7, nbinsx=30,
))
fig.add_trace(go.Histogram(
    x=anom, name="Anomalous Records", marker_color="#e74c3c",
    opacity=0.7, nbinsx=30,
))
fig.update_layout(
    title="Silver Layer: Quality Score Distribution (Clean vs Anomalous)",
    xaxis_title="Quality Score", yaxis_title="Count",
    barmode="overlay", template="plotly_white", height=400,
)
fig.show()

### Before vs After: What Silver Cleaned

Compare the raw Bronze data with the refined Silver output.

In [ ]:
comparison = pd.DataFrame({
    "Metric": [
        "Row Count",
        "Column Count",
        "Has Corrupt Records Column",
        "Has Anomaly Flags",
        "Has Z-Scores",
        "Has Quality Score",
    ],
    "Bronze": [
        f"{bronze_count:,}",
        str(len(bronze_cols)),
        str("_corrupt_record" in bronze_cols),
        "False",
        "False",
        "False",
    ],
    "Silver": [
        f"{silver_count:,}",
        str(len(silver_cols)),
        str("_corrupt_record" in silver_cols),
        str("_is_anomaly" in silver_cols),
        str("_temperature_zscore" in silver_cols),
        str("_quality_score" in silver_cols),
    ],
})
comparison

## 5. Gold Layer: Analytics-Ready Aggregations

Gold produces four purpose-built tables from Silver:

| Table | Purpose |
|---|---|
| `device_summary` | Per-device windowed sensor statistics |
| `anomaly_summary` | Anomaly breakdown by device and type |
| `device_health` | Composite health score with risk tiers |
| `ml_features` | Wide feature vector for model consumption |

In [ ]:
for name, info in gold_tables.items():
    df = info["df"]
    if df is None:
        print(f"\n=== {name}: NOT AVAILABLE ===")
        continue
    print(f"\n=== {name} ({info['count']} rows, {info['cols']} columns) ===")
    df.show(5, truncate=False)

In [ ]:
if gold_tables["device_summary"]["df"] is not None:
    summary_pdf = gold_tables["device_summary"]["df"].toPandas()

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Avg Temperature by Device", "Anomaly Rate by Device"],
    )
    top = summary_pdf.nlargest(15, "event_count")

    fig.add_trace(go.Bar(
        x=top["device_id"], y=top["avg_temperature"],
        marker_color=top["anomaly_rate"].apply(
            lambda r: "#e74c3c" if r > 0 else "#2ecc71"
        ),
        name="Avg Temp",
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=top["device_id"], y=top["anomaly_rate"],
        marker_color="#e67e22", name="Anomaly Rate",
    ), row=1, col=2)

    fig.update_layout(
        title="Gold Device Summary: Top 15 Devices by Event Count",
        template="plotly_white", height=400, showlegend=False,
    )
    fig.show()

## 6. Delta Lake Lineage: Version History

Every write to a Delta table creates a new version with full metadata.
This provides audit trail, rollback capability, and data lineage.

In [ ]:
for name, path in [("Bronze", bronze_path), ("Silver", silver_path)]:
    dt = DeltaTable.forPath(spark, path)
    hist = dt.history().select(
        "version", "timestamp", "operation",
        col("operationMetrics.numOutputRows").alias("rows_written"),
        col("operationMetrics.numOutputBytes").alias("bytes_written"),
    )
    print(f"\n=== {name} Delta History ===")
    hist.show(truncate=False)

### Time Travel: Reading Previous Versions

Delta Lake lets you query any previous version — useful for debugging,
auditing, and reproducible analysis.

In [ ]:
bronze_v0 = spark.read.format("delta").option("versionAsOf", 0).load(bronze_path)
bronze_latest = bronze_df

print(f"Bronze version 0:      {bronze_v0.count():,} rows")
print(f"Bronze latest version: {bronze_latest.count():,} rows")
print(f"Rows added since v0:   {bronze_latest.count() - bronze_v0.count():,}")

## 7. End-to-End Data Quality Summary

A single view showing how data quality improves across the medallion layers.

In [ ]:
bronze_corrupt = bronze_df.filter(col("_corrupt_record").isNotNull()).count()
bronze_nulls = bronze_df.filter(
    col("device_id").isNull() | col("timestamp").isNull()
).count()

silver_anomalies = silver_df.filter(col("_is_anomaly")).count()
silver_clean = silver_count - silver_anomalies
avg_quality = silver_df.select(avg("_quality_score")).first()[0]

summary = pd.DataFrame([
    {"Layer": "Bronze", "Metric": "Total Records", "Value": bronze_count},
    {"Layer": "Bronze", "Metric": "Corrupt Records", "Value": bronze_corrupt},
    {"Layer": "Bronze", "Metric": "Null-Key Records", "Value": bronze_nulls},
    {"Layer": "Silver", "Metric": "Clean Records", "Value": silver_clean},
    {"Layer": "Silver", "Metric": "Anomalous Records", "Value": silver_anomalies},
    {"Layer": "Silver", "Metric": "Avg Quality Score", "Value": round(avg_quality, 4)},
    {"Layer": "Gold", "Metric": "Aggregate Rows", "Value": gold_total},
])

fig = px.bar(
    summary, x="Metric", y="Value", color="Layer",
    color_discrete_map={"Bronze": "#cd7f32", "Silver": "#c0c0c0", "Gold": "#ffd700"},
    title="End-to-End Data Quality Summary",
)
fig.update_layout(template="plotly_white", height=400)
fig.show()

In [ ]:
spark.stop()